# 樹木資料集採樣品質評估與統計視覺化 (Sampling Quality EDA)
> **參考基準**：`simpleSam/segment_focusPrimary_subcanopy_withoutTrunk_leafPixel_nearbyCanopy_yolo_batch_fix.py`

本 Notebook 用於互動式探索 YOLO 資料集（`dataset/`）的採樣品質，包含：
1. **幾何尺度分佈（KDE / Histogram & 標準差 $\sigma$）**
2. **空間中心分佈（Class 0: 主樹 vs Class 1: 周邊樹 2D 散佈圖）**
3. **SAM 3 雙向提示詞（Positive 次冠層 + Negative 枝幹）評估**
4. **SAM 3 影像特徵餘弦相似度與連拍重複資料洩漏檢測**
5. **光照與超綠指數（ExG）分佈**

In [2]:
import sys
from pathlib import Path

# 載入專案根目錄
project_root = Path.cwd().parent if Path.cwd().name == "samplingAnalysis" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from samplingAnalysis.export_sampling_analytics import (
    load_dataset_metadata,
    parse_yolo_segmentation_labels,
    extract_spectral_and_luminance_stats,
    run_sam3_feature_and_confidence_analysis,
    export_distribution_graphs_and_report,
)

print("環境與模組載入成功！")

環境與模組載入成功！


## 1. 解析資料集多邊形與影像統計

In [3]:
yaml_path = project_root / "dataset/dataset.yaml"
meta = load_dataset_metadata(yaml_path)

train_records = parse_yolo_segmentation_labels(meta["train_lbl_dir"])
val_records = parse_yolo_segmentation_labels(meta["val_lbl_dir"])
train_spectral = extract_spectral_and_luminance_stats(meta["train_img_dir"])
val_spectral = extract_spectral_and_luminance_stats(meta["val_img_dir"])

all_img_paths = sorted(
    list(meta["train_img_dir"].glob("*.jpg"))
    + list(meta["train_img_dir"].glob("*.jpeg"))
    + list(meta["train_img_dir"].glob("*.png"))
    + list(meta["val_img_dir"].glob("*.jpg"))
    + list(meta["val_img_dir"].glob("*.jpeg"))
    + list(meta["val_img_dir"].glob("*.png"))
)

print(f"★ 訓練集樣本數: {len(train_records)} instances")
print(f"★ 驗證集樣本數: {len(val_records)} instances")
print(f"★ 總分析圖片數: {len(all_img_paths)} images")

★ 訓練集樣本數: 261 instances
★ 驗證集樣本數: 102 instances
★ 總分析圖片數: 19 images


## 2. 執行 SAM 3 雙向提示特徵提取與相似度計算 (含 Negative Prompt)

In [4]:
ckpt_path = project_root / "sam3.pt"
sim_matrix, sam_scores, trunk_ratios, image_names = run_sam3_feature_and_confidence_analysis(
    all_images=all_img_paths,
    checkpoint_path=ckpt_path,
    prompt="tree subcanopy",
    negative_prompt="tree trunk, tree branch",
    negative_threshold=0.155,
    device="cpu",  # 可切換為 mps 或 cuda
)

/Users/rickyho/Documents/github/sam/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/rickyho/Documents/github/sam/.venv/lib/python3.12/site-packages/sam3/model/sam3_video_inference.py:800: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  @torch.autocast(device_type="cuda", dtype=torch.bfloat16)
/Users/rickyho/Documents/github/sam/.venv/lib/python3.12/site-packages/sam3/model/sam3_video_inference.py:908: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  @torch.autocast(device_type="cuda", dtype=torch.bfloat16)
/Users/rickyho/Documents/github/sam/.venv/lib/python3.12/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Im

[SAM 3] 正在載入模型權重：/Users/rickyho/Documents/github/sam/sam3.pt (Device: cpu)...


## 3. 生成 5 大核心分佈圖表與匯出報告

In [5]:
output_dir = project_root / "samplingAnalysis/output"
report = export_distribution_graphs_and_report(
    train_records=train_records,
    val_records=val_records,
    train_spectral=train_spectral,
    val_spectral=val_spectral,
    sim_matrix=sim_matrix,
    sam_scores=sam_scores,
    trunk_ratios=trunk_ratios,
    image_names=image_names,
    output_dir=output_dir,
)

# 展示關鍵統計結果
print("分佈檢驗結果:", report["area_metrics"])
print("類別分佈比例:", report["class_distribution"])
print("SAM 3 品質指標 (含枝幹比):", report["sam3_quality_metrics"])


🎉 採樣品質分析完成！圖表與報告已儲存至：/Users/rickyho/Documents/github/sam/samplingAnalysis/output
★ 總樣本數: 19 (Train: 261 instances, Val: 102 instances)
★ 面積分佈 KS 檢定 p-value: 0.0652 (無明顯分佈偏移)
★ 木質枝幹平均佔比 (Negative Trunk Ratio): 13.75%
★ 連拍高相似度樣本組: 13 組
★ 診斷報告檔案: /Users/rickyho/Documents/github/sam/samplingAnalysis/output/sampling_summary_report.json
分佈檢驗結果: {'train_area_mean': 0.05460819290788506, 'train_area_std': 0.19432743986294101, 'val_area_mean': 0.0393700849712549, 'val_area_std': 0.1662130691514874, 'ks_statistic': 0.14998873112463376, 'ks_p_value': 0.06519587821354848, 'covariate_shift_detected': False}
類別分佈比例: {'primary_tree_count (Class 0)': 282, 'nearby_tree_count (Class 1)': 81, 'class_imbalance_ratio': 3.481481051669006}
SAM 3 品質指標 (含枝幹比): {'mean_confidence': 0.5487841963768005, 'std_confidence': 0.21234261813761593, 'mean_trunk_to_canopy_ratio (木質枝幹佔比)': 0.1375249526802337, 'std_trunk_to_canopy_ratio': 0.22571755108221855, 'redundant_burst_pairs_count': 13, 'redundant_pairs': [{'image_a':